# Lab 01 — Exercise 6: Prediction verification

Notebook này kiểm chứng dự đoán sau khi đã ghi câu trả lời bằng tay. Corpus gồm D1 = medical image classification, D2 = medical image analysis, D3 = natural language processing; query là medical image classification.

So sánh cosine similarity của hai representation: TF-IDF với $idf(t) = \log(N / df(t))$ và raw count vector.

In [1]:
from collections import Counter
from math import log, sqrt

DOCUMENTS = {
    "D1": "medical image classification",
    "D2": "medical image analysis",
    "D3": "natural language processing",
}
QUERY = "medical image classification"
DISPLAY_PRECISION = 6
EXPECTED_RANKING = ("D1", "D2", "D3")
ZERO_SCORE = 0.0


In [2]:
tokenized_documents = {
    label: tuple(text.lower().split())
    for label, text in DOCUMENTS.items()
}
query_tokens = tuple(QUERY.lower().split())
document_frequencies = Counter(
    term
    for document in tokenized_documents.values()
    for term in set(document)
)
document_count = len(tokenized_documents)
idf_values = {
    term: log(document_count / frequency)
    for term, frequency in document_frequencies.items()
}
tf_idf_documents = {
    label: {
        term: (count / len(tokens)) * idf_values[term]
        for term, count in Counter(tokens).items()
    }
    for label, tokens in tokenized_documents.items()
}
query_tf_idf = {
    term: (count / len(query_tokens)) * idf_values.get(term, ZERO_SCORE)
    for term, count in Counter(query_tokens).items()
}
count_documents = {
    label: {term: float(count) for term, count in Counter(tokens).items()}
    for label, tokens in tokenized_documents.items()
}
query_count = {term: float(count) for term, count in Counter(query_tokens).items()}


In [3]:
tf_idf_scores = {
    label: sum(
        weight * query_tf_idf.get(term, ZERO_SCORE)
        for term, weight in vector.items()
    )
    / (
        sqrt(sum(weight * weight for weight in vector.values()))
        * sqrt(sum(weight * weight for weight in query_tf_idf.values()))
    )
    for label, vector in tf_idf_documents.items()
}
count_scores = {
    label: sum(
        weight * query_count.get(term, ZERO_SCORE)
        for term, weight in vector.items()
    )
    / (
        sqrt(sum(weight * weight for weight in vector.values()))
        * sqrt(sum(weight * weight for weight in query_count.values()))
    )
    for label, vector in count_documents.items()
}
tf_idf_ranking = tuple(sorted(tf_idf_scores.items(), key=lambda item: (-item[1], item[0])))
count_ranking = tuple(sorted(count_scores.items(), key=lambda item: (-item[1], item[0])))
idf_text = ", ".join(
    f"{term}={idf_values[term]:.{DISPLAY_PRECISION}f}"
    for term in sorted(idf_values)
)
tf_idf_rows = tuple(
    f"{position}. {label}: {score:.{DISPLAY_PRECISION}f}"
    for position, (label, score) in enumerate(tf_idf_ranking, start=1)
)
count_rows = tuple(
    f"{position}. {label}: {score:.{DISPLAY_PRECISION}f}"
    for position, (label, score) in enumerate(count_ranking, start=1)
)

print(f"IDF values: {idf_text}")
print("\nTF-IDF cosine similarity")
print("\n".join(tf_idf_rows))
print("\nCount-vector cosine similarity")
print("\n".join(count_rows))

assert tuple(label for label, _ in tf_idf_ranking) == EXPECTED_RANKING
assert tuple(label for label, _ in count_ranking) == EXPECTED_RANKING
assert tf_idf_ranking[1][1] < count_ranking[1][1]


IDF values: analysis=1.098612, classification=1.098612, image=0.405465, language=1.098612, medical=0.405465, natural=1.098612, processing=1.098612

TF-IDF cosine similarity
1. D1: 1.000000
2. D2: 0.214099
3. D3: 0.000000

Count-vector cosine similarity
1. D1: 1.000000
2. D2: 0.666667
3. D3: 0.000000


## Kiểm chứng

Cả hai representation đều cho D1 > D2 > D3. Với TF-IDF, classification có IDF cao vì chỉ xuất hiện trong D1, nên D2 bị giảm điểm mạnh hơn so với count vector.